In [0]:
# Set storage account key
# Replace YOUR_STORAGE_ACCOUNT_KEY with your own key before running

spark.conf.set(
"fs.azure.account.key.weatherdatadatalake.dfs.core.windows.net",
"YOUR_STORAGE_ACCOUNT_KEY"
)

In [0]:
df = spark.read.json(
"abfss://bronze@weatherdatadatalake.dfs.core.windows.net/weather/"
)

display(df)

base,clouds,cod,coord,dt,id,main,name,sys,timezone,visibility,weather,wind
stations,List(75),200,"List(28.6667, 77.2167)",1773135221,1273294,"List(33.45, 986, 8, 1011, 1011, 36.34, 36.34, 36.34)",Delhi,"List(IN, 1773104825, 1773147371)",19800,10000,"List(List(broken clouds, 04d, 803, Clouds))","List(265, 4.12, 1.76)"
stations,List(0),200,"List(12.9762, 77.6033)",1773135130,1277333,"List(28.44, 914, 16, 1010, 1010, 30.36, 30.36, 30.36)",Bengaluru,"List(IN, 1773104434, 1773147576)",19800,10000,"List(List(clear sky, 01d, 800, Clear))","List(104, 6.33, 5.27)"
stations,List(0),200,"List(17.3753, 78.4744)",1773135198,1269843,"List(31.58, 948, 26, 1011, 1011, 33.08, 33.08, 33.08)",Hyderabad,"List(IN, 1773104304, 1773147288)",19800,10000,"List(List(clear sky, 01d, 800, Clear))","List(97, 5.23, 4.67)"
stations,List(3),200,"List(19.0144, 72.8479)",1773135453,1275339,"List(31.32, 1008, 53, 1009, 1009, 29.86, 29.86, 29.86)",Mumbai,"List(IN, 1773105684, 1773148608)",19800,10000,"List(List(clear sky, 01d, 800, Clear))","List(297, 8.18, 6.88)"
stations,List(1),200,"List(18.5196, 73.8553)",1773135000,1259229,"List(33.47, 938, 15, 1010, 1010, 35.89, 35.89, 35.89)",Pune,"List(IN, 1773105433, 1773148376)",19800,10000,"List(List(clear sky, 01d, 800, Clear))","List(84, 6.09, 5.04)"


In [0]:
df.printSchema()

root
 |-- base: string (nullable = true)
 |-- clouds: struct (nullable = true)
 |    |-- all: long (nullable = true)
 |-- cod: long (nullable = true)
 |-- coord: struct (nullable = true)
 |    |-- lat: double (nullable = true)
 |    |-- lon: double (nullable = true)
 |-- dt: long (nullable = true)
 |-- id: long (nullable = true)
 |-- main: struct (nullable = true)
 |    |-- feels_like: double (nullable = true)
 |    |-- grnd_level: long (nullable = true)
 |    |-- humidity: long (nullable = true)
 |    |-- pressure: long (nullable = true)
 |    |-- sea_level: long (nullable = true)
 |    |-- temp: double (nullable = true)
 |    |-- temp_max: double (nullable = true)
 |    |-- temp_min: double (nullable = true)
 |-- name: string (nullable = true)
 |-- sys: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- sunrise: long (nullable = true)
 |    |-- sunset: long (nullable = true)
 |-- timezone: long (nullable = true)
 |-- visibility: long (nullable = true)
 |--

In [0]:
from pyspark.sql.functions import col

silver_df = df.select(
    col("name").alias("city"),
    col("main.temp").alias("temperature"),
    col("main.humidity").alias("humidity"),
    col("weather")[0]["main"].alias("weather_condition")
)

display(silver_df)

city,temperature,humidity,weather_condition
Delhi,36.34,8,Clouds
Bengaluru,30.36,16,Clear
Hyderabad,33.08,26,Clear
Mumbai,29.86,53,Clear
Pune,35.89,15,Clear


In [0]:
silver_df.write.format("delta").mode("overwrite").save(
"abfss://silver@weatherdatadatalake.dfs.core.windows.net/silver/weather/"
)

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_weather")

In [0]:
%sql
select * from silver_weather

city,temperature,humidity,weather_condition
Bengaluru,30.36,16,Clear
Delhi,36.34,8,Clouds
Hyderabad,33.08,26,Clear
Mumbai,29.86,53,Clear
Pune,35.89,15,Clear
